# Control de Calidad - Planta de Harina y Aceite de Pescado
## Estadística Descriptiva aplicada a Negocios (Módulo 1)
**Supervisora de Calidad — Preparación Big Data Analytics Centrum PUCP**

---
## 1. Importar librerías y crear datos de calidad

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Datos simulados de 60 lotes de producción
np.random.seed(42)
n = 60

datos = {
    'fecha': pd.date_range('2025-01-01', periods=n, freq='D'),
    'turno': np.tile(['Mañana', 'Tarde', 'Noche'], n // 3),
    'linea': np.tile(['Línea 1', 'Línea 2'], n // 2),
    'proteina_pct': np.random.normal(loc=67.0, scale=1.5, size=n).round(2),   # % proteína (spec: 65-70%)
    'humedad_pct':  np.random.normal(loc=10.0, scale=0.8, size=n).round(2),   # % humedad (spec: máx 12%)
    'grasa_pct':    np.random.normal(loc=9.5,  scale=1.0, size=n).round(2),   # % grasa (spec: máx 12%)
    'tvn':          np.random.normal(loc=90.0, scale=15.0, size=n).round(1),  # TVN mg/100g (spec: máx 120)
    'acidez_ffa':   np.random.normal(loc=3.5,  scale=0.5, size=n).round(2),   # FFA aceite % (spec: máx 5%)
}

df = pd.DataFrame(datos)
print(f'Dataset: {df.shape[0]} lotes, {df.shape[1]} variables')
df.head(10)

ModuleNotFoundError: No module named 'seaborn'

---
## 2. Estadística Descriptiva básica
Primero entendemos el comportamiento general de cada parámetro de calidad.

In [2]:
# describe() da: count, mean, std, min, 25%, 50%, 75%, max
parametros = ['proteina_pct', 'humedad_pct', 'grasa_pct', 'tvn', 'acidez_ffa']
df[parametros].describe().round(2)

NameError: name 'df' is not defined

In [ ]:
# ¿Cuántos lotes están FUERA de especificación?
fuera_spec = {
    'proteina_fuera_spec': ((df.proteina_pct < 65) | (df.proteina_pct > 70)).sum(),
    'humedad_fuera_spec':  (df.humedad_pct > 12).sum(),
    'grasa_fuera_spec':    (df.grasa_pct > 12).sum(),
    'tvn_fuera_spec':      (df.tvn > 120).sum(),
    'acidez_fuera_spec':   (df.acidez_ffa > 5).sum(),
}

for param, cantidad in fuera_spec.items():
    pct = (cantidad / n) * 100
    print(f'{param}: {cantidad} lotes ({pct:.1f}%)')

---
## 3. Visualización: Distribución de parámetros de calidad

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribución de Parámetros de Calidad — Harina de Pescado', fontsize=14, fontweight='bold')

configs = [
    ('proteina_pct', '% Proteína',  'green',  65, 70),
    ('humedad_pct',  '% Humedad',   'blue',   None, 12),
    ('grasa_pct',    '% Grasa',     'orange', None, 12),
    ('tvn',          'TVN mg/100g', 'red',    None, 120),
    ('acidez_ffa',   'FFA Aceite %','purple', None, 5),
]

for i, (col, titulo, color, lim_min, lim_max) in enumerate(configs):
    ax = axes[i // 3][i % 3]
    ax.hist(df[col], bins=12, color=color, alpha=0.7, edgecolor='black')
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Media: {df[col].mean():.2f}')
    if lim_min:
        ax.axvline(lim_min, color='red', linestyle=':', linewidth=2, label=f'Límite inf: {lim_min}')
    if lim_max:
        ax.axvline(lim_max, color='red', linestyle=':', linewidth=2, label=f'Límite sup: {lim_max}')
    ax.legend(fontsize=8)

axes[1][2].set_visible(False)
plt.tight_layout()
plt.savefig('calidad_distribuciones.png', dpi=120, bbox_inches='tight')
plt.show()
print('Gráfico guardado como calidad_distribuciones.png')

---
## 4. Análisis por Turno (groupby)
¿En qué turno la calidad es mejor o peor?

In [ ]:
resumen_turno = df.groupby('turno')[parametros].agg(['mean', 'std']).round(2)
resumen_turno

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Comparación de Calidad por Turno', fontsize=13, fontweight='bold')

# Boxplot proteína por turno
df.boxplot(column='proteina_pct', by='turno', ax=axes[0], 
           boxprops=dict(color='green'), medianprops=dict(color='red', linewidth=2))
axes[0].set_title('% Proteína por Turno')
axes[0].set_xlabel('Turno')
axes[0].set_ylabel('% Proteína')
axes[0].axhline(65, color='red', linestyle='--', alpha=0.5, label='Límite inferior')
axes[0].axhline(70, color='red', linestyle='--', alpha=0.5, label='Límite superior')
axes[0].legend()

# Boxplot TVN por turno
df.boxplot(column='tvn', by='turno', ax=axes[1],
           boxprops=dict(color='blue'), medianprops=dict(color='red', linewidth=2))
axes[1].set_title('TVN por Turno')
axes[1].set_xlabel('Turno')
axes[1].set_ylabel('TVN mg/100g')
axes[1].axhline(120, color='red', linestyle='--', alpha=0.5, label='Límite máximo')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 5. Correlación entre parámetros
¿Qué parámetros se relacionan entre sí? Esto es clave para Machine Learning.

In [ ]:
correlacion = df[parametros].corr().round(2)

plt.figure(figsize=(8, 6))
sns.heatmap(correlacion, 
            annot=True, 
            cmap='RdYlGn', 
            vmin=-1, vmax=1,
            linewidths=0.5,
            fmt='.2f')
plt.title('Correlación entre Parámetros de Calidad', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlacion_calidad.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nInterpretación:')
print('  +1.0 = correlación perfecta positiva')
print('  -1.0 = correlación perfecta negativa')
print('   0.0 = sin correlación')

---
## 6. Gráfico de Control — Serie de Tiempo de Proteína
Visualiza la variación de proteína a lo largo del tiempo con límites de control.

In [ ]:
media = df.proteina_pct.mean()
desv  = df.proteina_pct.std()
lcl   = media - 3 * desv   # Límite de control inferior (3 sigma)
ucl   = media + 3 * desv   # Límite de control superior (3 sigma)

plt.figure(figsize=(14, 5))
plt.plot(df.fecha, df.proteina_pct, 'b-o', markersize=4, linewidth=1, label='% Proteína')
plt.axhline(media, color='green',  linewidth=2,   linestyle='-',  label=f'Media: {media:.2f}%')
plt.axhline(ucl,   color='red',    linewidth=1.5, linestyle='--', label=f'UCL (3σ): {ucl:.2f}%')
plt.axhline(lcl,   color='red',    linewidth=1.5, linestyle='--', label=f'LCL (3σ): {lcl:.2f}%')
plt.axhline(70,    color='orange', linewidth=1,   linestyle=':',  label='Esp. máx: 70%')
plt.axhline(65,    color='orange', linewidth=1,   linestyle=':',  label='Esp. mín: 65%')

# Marcar puntos fuera de control
fuera = df[(df.proteina_pct > ucl) | (df.proteina_pct < lcl)]
plt.scatter(fuera.fecha, fuera.proteina_pct, color='red', s=80, zorder=5, label=f'Fuera de control: {len(fuera)}')

plt.title('Gráfico de Control — % Proteína en Harina de Pescado', fontsize=13, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('% Proteína')
plt.xticks(rotation=45)
plt.legend(loc='lower right', fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('grafico_control_proteina.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Lotes fuera de control estadístico: {len(fuera)}')

---
## 7. Resumen ejecutivo para gerencia

In [ ]:
print('=' * 55)
print('   REPORTE DE CALIDAD — HARINA DE PESCADO')
print('=' * 55)
print(f'  Período analizado: {df.fecha.min().date()} al {df.fecha.max().date()}')
print(f'  Total de lotes:    {n}')
print()
print(f'  Proteína promedio: {df.proteina_pct.mean():.2f}% ± {df.proteina_pct.std():.2f}')
print(f'  Humedad promedio:  {df.humedad_pct.mean():.2f}% ± {df.humedad_pct.std():.2f}')
print(f'  Grasa promedio:    {df.grasa_pct.mean():.2f}% ± {df.grasa_pct.std():.2f}')
print(f'  TVN promedio:      {df.tvn.mean():.1f} ± {df.tvn.std():.1f} mg/100g')
print(f'  FFA aceite prom.:  {df.acidez_ffa.mean():.2f}% ± {df.acidez_ffa.std():.2f}')
print()
print('  LOTES FUERA DE ESPECIFICACIÓN:')
for param, cantidad in fuera_spec.items():
    pct = (cantidad / n) * 100
    estado = '✓ OK' if cantidad == 0 else '⚠ REVISAR'
    print(f'  {estado}  {param}: {cantidad} lotes ({pct:.1f}%)')
print('=' * 55)

---
## Ejercicio para practicar

1. **Agrega una columna** `aprobado` que sea `True` si el lote cumple TODOS los parámetros de calidad.
2. **Calcula el % de aprobación** por línea de producción.
3. **Crea un gráfico de barras** que compare el promedio de proteína entre Línea 1 y Línea 2.

*Pista para el punto 1:*
```python
df['aprobado'] = (
    (df.proteina_pct >= 65) & (df.proteina_pct <= 70) &
    (df.humedad_pct <= 12) &
    ...
)
```